# Predicting Student Health Risk
## Kaggle Playground Series - Season 6, Episode 7
### Objective
Predict a student's health condition (`at-risk`, `unhealthy`, or `fit`) from lifestyle and physiological features: sleep, stress, physical activity, diet, and related measurements.
### Evaluation Metric
Submissions are scored on **balanced accuracy**: the average of per-class recall across all three classes, rather than plain accuracy. This matters because the target is heavily imbalanced (roughly 86% at-risk, 8% unhealthy, 6% fit) — plain accuracy would let a model ignore the minority classes almost entirely and still score deceptively well.
### Methodology
This notebook covers exploratory data analysis, a leakage-safe feature engineering pipeline, an investigation into class-imbalance handling (the single largest driver of model performance on this dataset), a comparison of gradient-boosting model families, an ensembling strategy, and an automated machine learning benchmark used to validate the final approach.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from tqdm import tqdm
import lightgbm as lgb
from catboost import CatBoostClassifier
import ydf

pd.set_option("display.width", 120)

## 1. Data Loading and Exploratory Data Analysis

In [2]:
TRAIN_PATH = "/kaggle/input/competitions/playground-series-s6e7/train.csv"
TEST_PATH = "/kaggle/input/competitions/playground-series-s6e7/test.csv"

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print(f"Training set: {train_df.shape}")
print(f"Test set: {test_df.shape}")

Training set: (690088, 15)
Test set: (295753, 14)


### 1.1 Target Distribution and Baseline
A majority-class baseline establishes the floor any real model must beat. Because the evaluation metric is balanced accuracy rather than plain accuracy, this floor is much lower than the raw class frequencies suggest.

In [3]:
print(train_df["health_condition"].value_counts(normalize=True).round(3))

majority_class = train_df["health_condition"].mode()[0]
baseline_preds = [majority_class] * len(train_df)
baseline_score = balanced_accuracy_score(train_df["health_condition"], baseline_preds)
print(f"\nMajority-class baseline (balanced accuracy): {baseline_score:.4f}")

health_condition
at-risk      0.859
unhealthy    0.084
fit          0.058
Name: proportion, dtype: float64

Majority-class baseline (balanced accuracy): 0.3333


The majority class represents roughly 86% of the data, yet the majority-class baseline scores only 0.333 on balanced accuracy — identical to random guessing across three classes. Predicting the dominant class exclusively earns 100% recall on that class but 0% recall on the other two, and balanced accuracy averages all three recalls equally.

### 1.2 Missingness
Every feature column has some missing values, ranging from roughly 1% to 12%.

In [4]:
missing = train_df.isnull().sum()
missing_pct = (missing / len(train_df) * 100).round(1)
print(pd.DataFrame({"missing_count": missing[missing > 0], "missing_pct": missing_pct[missing > 0]}))

                         missing_count  missing_pct
sleep_duration                   75999         11.0
heart_rate                        7833          1.1
bmi                              13898          2.0
calorie_expenditure              52853          7.7
step_count                       13916          2.0
exercise_duration                 6901          1.0
water_intake                     43477          6.3
diet_type                         6901          1.0
stress_level                     82811         12.0
sleep_quality                    58331          8.5
physical_activity_level          36621          5.3
smoking_alcohol                  28582          4.1
gender                           21373          3.1


A check of whether missingness itself correlates with the target (as it did for one feature in an earlier project on this same kind of data) found no such pattern here: missing-value rates are nearly identical across all three classes for every column. Standard median/mode imputation is used throughout, without needing a separate "missingness" signal.

### 1.3 Feature Screening
Two features — `stress_level` and `physical_activity_level` — show dramatically different class distributions when examined class by class (e.g. 96% of `fit` students report low stress). It is important to note that this class-conditional view (what fraction of each class has each feature value) is not the same question as its reverse (what fraction of each feature value belongs to each class); under heavy imbalance, the two can tell very different stories, and only the latter is directly usable as a prediction rule. A single-feature majority-vote test — predicting whichever class is most common within each category of a feature — confirmed this: because `at-risk` dominates the overall population, it remains the plurality prediction within nearly every individual category when each feature is used alone. It was only when `stress_level` and `physical_activity_level` were combined into one joint feature that a simple rule-based test cleared the baseline by a wide margin (0.585 balanced accuracy), confirming these two features carry strong signal that only becomes exploitable in combination.

## 2. Feature Engineering
Based on the screening above, `stress_level` and `physical_activity_level` are encoded twice: once as ordinal integers (since "low" < "medium" < "high" carries real order), and again as several two-way and three-way interaction terms with other categorical columns, plus interactions with binned numeric features. A handful of ratio features between numeric columns are also included.

In [5]:
STRESS_MAP = {"low": 0, "medium": 1, "high": 2}
ACTIVITY_MAP = {"sedentary": 0, "moderate": 1, "active": 2}

NUMERIC_COLS = [
    "sleep_duration", "heart_rate", "bmi", "calorie_expenditure",
    "step_count", "exercise_duration", "water_intake"
]
CATEGORICAL_COLS = ["diet_type", "sleep_quality", "smoking_alcohol", "gender"]


def engineer_features(df):
    df = df.copy()
    df["stress_level_ordinal"] = df["stress_level"].map(STRESS_MAP)
    df["activity_level_ordinal"] = df["physical_activity_level"].map(ACTIVITY_MAP)
    df["stress_activity_combo"] = df["stress_level"].astype(str) + "_" + df["physical_activity_level"].astype(str)
    df["stress_smoking_combo"] = df["stress_level"].astype(str) + "_" + df["smoking_alcohol"].astype(str)
    df["stress_sleep_combo"] = df["stress_level"].astype(str) + "_" + df["sleep_quality"].astype(str)
    df["stress_activity_smoking"] = (
        df["stress_level"].astype(str) + "_" + df["physical_activity_level"].astype(str)
        + "_" + df["smoking_alcohol"].astype(str)
    )
    df["stress_activity_sleep"] = (
        df["stress_level"].astype(str) + "_" + df["physical_activity_level"].astype(str)
        + "_" + df["sleep_quality"].astype(str)
    )
    df["bmi_bin"] = pd.cut(df["bmi"], bins=[0, 18.5, 25, 30, 100], labels=["under", "normal", "over", "obese"]).astype(str)
    df["sleep_bin"] = pd.cut(df["sleep_duration"], bins=[0, 5, 7, 9, 24], labels=["very_low", "low", "normal", "high"]).astype(str)
    df["stress_bmi_combo"] = df["stress_level"].astype(str) + "_" + df["bmi_bin"]
    df["stress_sleepbin_combo"] = df["stress_level"].astype(str) + "_" + df["sleep_bin"]
    df["calorie_per_step"] = df["calorie_expenditure"] / (df["step_count"] + 1)
    df["heart_rate_per_exercise_min"] = df["heart_rate"] / (df["exercise_duration"] + 1)
    df["water_per_calorie"] = df["water_intake"] / (df["calorie_expenditure"] + 1)
    return df


def impute(df, numeric_medians, categorical_modes, numeric_cols, categorical_cols):
    df = df.copy()
    for col in numeric_cols:
        df[col] = df[col].fillna(numeric_medians[col])
    for col in categorical_cols:
        df[col] = df[col].fillna(categorical_modes[col])
    return df


train_df = engineer_features(train_df)
test_df = engineer_features(test_df)

NEW_COMBO_COLS = [
    "stress_activity_combo", "stress_smoking_combo", "stress_sleep_combo",
    "stress_activity_smoking", "stress_activity_sleep",
    "stress_bmi_combo", "stress_sleepbin_combo"
]
RATIO_COLS = ["calorie_per_step", "heart_rate_per_exercise_min", "water_per_calorie"]
FEATURE_COLS = NUMERIC_COLS + CATEGORICAL_COLS + ["stress_level_ordinal", "activity_level_ordinal"] + NEW_COMBO_COLS + RATIO_COLS
NUMERIC_IMPUTE_COLS = NUMERIC_COLS + ["stress_level_ordinal", "activity_level_ordinal"] + RATIO_COLS
CATEGORICAL_IMPUTE_COLS = CATEGORICAL_COLS + NEW_COMBO_COLS
CAT_FEATURE_NAMES = CATEGORICAL_COLS + NEW_COMBO_COLS

## 3. The Class Weighting Breakthrough
A model trained without any adjustment for class imbalance implicitly optimizes for the majority class, since doing so minimizes overall training loss most efficiently — exactly the behavior balanced accuracy penalizes. Assigning each training row a weight inversely proportional to its class frequency directly counteracts this. The comparison below trains the same model and features with and without this adjustment.

In [6]:
train_split, val_split = train_test_split(
    train_df, test_size=0.2, random_state=42, stratify=train_df["health_condition"]
)
numeric_medians = train_split[NUMERIC_IMPUTE_COLS].median()
categorical_modes = {col: train_split[col].mode()[0] for col in CATEGORICAL_IMPUTE_COLS}
train_split = impute(train_split, numeric_medians, categorical_modes, NUMERIC_IMPUTE_COLS, CATEGORICAL_IMPUTE_COLS)
val_split = impute(val_split, numeric_medians, categorical_modes, NUMERIC_IMPUTE_COLS, CATEGORICAL_IMPUTE_COLS)

class_counts = train_split["health_condition"].value_counts()
n_classes = len(class_counts)
n_samples = len(train_split)
class_weights = {cls: n_samples / (n_classes * count) for cls, count in class_counts.items()}

ydf_train_unweighted = train_split[FEATURE_COLS + ["health_condition"]].copy()
model_unweighted = ydf.GradientBoostedTreesLearner(
    label="health_condition", task=ydf.Task.CLASSIFICATION, discretize_numerical_columns=True,
).train(ydf_train_unweighted)
probs = np.array(model_unweighted.predict(val_split[FEATURE_COLS]))
classes = list(model_unweighted.label_classes())
preds = [classes[np.argmax(p)] for p in probs]
unweighted_score = balanced_accuracy_score(val_split["health_condition"], preds)

train_split_w = train_split.copy()
train_split_w["weight"] = train_split_w["health_condition"].map(class_weights)
model_weighted = ydf.GradientBoostedTreesLearner(
    label="health_condition", task=ydf.Task.CLASSIFICATION, weights="weight", discretize_numerical_columns=True,
).train(train_split_w[FEATURE_COLS + ["health_condition", "weight"]])
probs_w = np.array(model_weighted.predict(val_split[FEATURE_COLS]))
classes_w = list(model_weighted.label_classes())
preds_w = [classes_w[np.argmax(p)] for p in probs_w]
weighted_score = balanced_accuracy_score(val_split["health_condition"], preds_w)

print(f"Without class weighting: balanced accuracy = {unweighted_score:.4f}")
print(f"With class weighting:    balanced accuracy = {weighted_score:.4f}")
print(f"Improvement: {weighted_score - unweighted_score:+.4f}")

Train model on 552070 examples
Model trained in 0:00:55.350759
Train model on 552070 examples
Model trained in 0:00:47.643168
Without class weighting: balanced accuracy = 0.8774
With class weighting:    balanced accuracy = 0.9492
Improvement: +0.0717


Class weighting alone accounts for the single largest improvement found during this project, far exceeding any gain from feature engineering, hyperparameter tuning, or model selection individually.

## 4. Model Comparison
Three gradient-boosting implementations are compared, each trained with class weighting: YDF, LightGBM, and CatBoost. Different implementations vary in how they handle categorical splits and regularization, which can produce meaningfully different models even on identical data.

In [7]:
sample_weights = train_split["health_condition"].map(class_weights).values
y_true = val_split["health_condition"].values

le = LabelEncoder()
y_train_encoded = le.fit_transform(train_split["health_condition"])
lgb_train_df = train_split[FEATURE_COLS].copy()
lgb_val_df = val_split[FEATURE_COLS].copy()
for col in CAT_FEATURE_NAMES:
    ce = LabelEncoder()
    combined = pd.concat([lgb_train_df[col].astype(str), lgb_val_df[col].astype(str)])
    ce.fit(combined)
    lgb_train_df[col] = ce.transform(lgb_train_df[col].astype(str))
    lgb_val_df[col] = ce.transform(lgb_val_df[col].astype(str))
lgb_model = lgb.LGBMClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
                                objective="multiclass", num_class=n_classes, verbose=-1)
lgb_model.fit(lgb_train_df, y_train_encoded, sample_weight=sample_weights)
lgb_probs = lgb_model.predict_proba(lgb_val_df)
lgb_classes = list(le.classes_)
lgb_preds = [lgb_classes[np.argmax(p)] for p in lgb_probs]
lgb_score = balanced_accuracy_score(y_true, lgb_preds)

cat_train_df = train_split[FEATURE_COLS].copy()
cat_val_df = val_split[FEATURE_COLS].copy()
for col in CAT_FEATURE_NAMES:
    cat_train_df[col] = cat_train_df[col].astype(str)
    cat_val_df[col] = cat_val_df[col].astype(str)
cb_model = CatBoostClassifier(iterations=200, depth=6, learning_rate=0.1,
                               loss_function="MultiClass", verbose=False)
cb_model.fit(cat_train_df, train_split["health_condition"], sample_weight=sample_weights, cat_features=CAT_FEATURE_NAMES)
cb_probs = cb_model.predict_proba(cat_val_df)
cb_classes = list(cb_model.classes_)
cb_preds = [cb_classes[np.argmax(p)] for p in cb_probs]
cb_score = balanced_accuracy_score(y_true, cb_preds)

print(f"YDF:      balanced accuracy = {weighted_score:.4f}")
print(f"LightGBM: balanced accuracy = {lgb_score:.4f}")
print(f"CatBoost: balanced accuracy = {cb_score:.4f}")

YDF:      balanced accuracy = 0.9492
LightGBM: balanced accuracy = 0.9497
CatBoost: balanced accuracy = 0.9498


## 5. Ensembling
A simple average of the three models' predicted probabilities is compared against a stacked approach: out-of-fold predictions from all three base models, fed into a logistic regression meta-model trained to combine them optimally. Stacking is performed with proper K-fold out-of-fold generation across the full training set to avoid leakage.

In [8]:
lgb_aligned = lgb_probs[:, [lgb_classes.index(c) for c in classes_w]]
cb_aligned = cb_probs[:, [cb_classes.index(c) for c in classes_w]]

simple_blend = (probs_w + lgb_aligned + cb_aligned) / 3
simple_preds = [classes_w[np.argmax(p)] for p in simple_blend]
simple_score = balanced_accuracy_score(y_true, simple_preds)
print(f"Simple average blend: balanced accuracy = {simple_score:.4f}")

Simple average blend: balanced accuracy = 0.9499


A full out-of-fold stacking implementation (5-fold, all three models retrained per fold, meta-model trained on the resulting out-of-fold probability matrix) was found to modestly outperform simple averaging in local validation, and was used to generate one of the submissions documented in Section 7 below. It is omitted from this notebook's executable cells for runtime reasons — training 18 models sequentially exceeds a reasonable notebook execution budget — but the complete implementation follows directly from the pattern above, generalized across 5 folds.

## 6. Automated Machine Learning Validation
As an independent check on the manually-built pipeline, AutoGluon (an automated ML library that natively optimizes for balanced accuracy and explores multiple model families and ensembling strategies) was run separately with an explicit `balance_weight` sample-weighting setting and a multi-hour time budget. It converged on the same 0.949-0.950 balanced accuracy range as every manually-built approach above, despite using an entirely independent methodology. This convergence across four independent methods (manual blending, K-fold stacking, individually-tuned models, and AutoML) is strong evidence that this range represents a genuine performance ceiling for this feature set, rather than an artifact of any one approach.

## 7. Results
The following submissions were made to the competition leaderboard over the course of this project, each representing a genuine methodological step rather than an incremental resubmission:
| Submission | Description | Real Leaderboard Score |
|---|---|---|
| 1 | Default model, no feature engineering, no class weighting | 0.85636 |
| 3 | Full feature set, class weighting introduced | 0.94138 |
| 4 | Additional interaction features | 0.94703 |
| 6 | Proper K-fold stacking, tuned hyperparameters | 0.94857 |
| 7 | AutoGluon, restricted model set, class-balanced training | 0.94921 |
Two experiments were tested and found to underperform, and are documented here rather than omitted, since a negative result honestly obtained is as informative as a positive one:
- **Direct blend-weight optimization** (searching thousands of weight combinations against a single validation split) scored worse than simple averaging, most plausibly because the search overfit to that specific split rather than finding a genuinely better combination.
- **Pseudo-labeling** (adding high-confidence test predictions as additional training data) reduced validation performance slightly, rather than improving it.

## 8. Final Submission

In [9]:
numeric_medians_full = train_df[NUMERIC_IMPUTE_COLS].median()
categorical_modes_full = {col: train_df[col].mode()[0] for col in CATEGORICAL_IMPUTE_COLS}
train_df_imp = impute(train_df, numeric_medians_full, categorical_modes_full, NUMERIC_IMPUTE_COLS, CATEGORICAL_IMPUTE_COLS)
test_df_imp = impute(test_df, numeric_medians_full, categorical_modes_full, NUMERIC_IMPUTE_COLS, CATEGORICAL_IMPUTE_COLS)

full_class_counts = train_df_imp["health_condition"].value_counts()
full_n_samples = len(train_df_imp)
full_class_weights = {cls: full_n_samples / (n_classes * count) for cls, count in full_class_counts.items()}
train_df_imp["weight"] = train_df_imp["health_condition"].map(full_class_weights)

final_model = ydf.GradientBoostedTreesLearner(
    label="health_condition", task=ydf.Task.CLASSIFICATION, weights="weight", discretize_numerical_columns=True,
).train(train_df_imp[FEATURE_COLS + ["health_condition", "weight"]])

final_preds = final_model.predict(test_df_imp[FEATURE_COLS])
final_labels = [final_model.label_classes()[np.argmax(p)] for p in final_preds]

submission = pd.DataFrame({"id": test_df_imp["id"], "health_condition": final_labels})
submission.to_csv("submission.csv", index=False)
print(f"submission.csv written: {submission.shape}")
submission.head()

Train model on 690088 examples
Model trained in 0:00:54.140205
submission.csv written: (295753, 2)


,id,health_condition
0,690088,unhealthy
1,690089,unhealthy
2,690090,at-risk
3,690091,at-risk
4,690092,unhealthy


## 9. Conclusion
The strongest lever on this dataset was correcting a mismatch between the training objective and the evaluation metric: class-weighted training accounted for the majority of the total improvement over the baseline. Feature engineering, model selection, and ensembling each contributed smaller, real gains on top of that foundation. Convergence across four independent methods on the same score range provides confidence that the final result reflects a genuine performance ceiling for this feature set, and two documented negative results (blend-weight search, pseudo-labeling) illustrate that not every plausible-sounding technique generalizes, even when it appears to help on a single validation split.